# LSTM Time Series - Stock Price Prediction
## Part 3 - Model Training
In this notebook, we import the scaled dataset files, prepare them in a format suitable for LSTM modeling, and proceed to train the LSTM model.

> **INPUT**: Scaled dataset files for training, validation, and testing periods, as processed in the preceding phase. <br/>
> **OUTPUT**: Trained LSTM model and analysis of its performance.

### 1. INITIALIZATION

In [1]:
import sys
sys.path.append('/Users/rifatordulu/Developer/lstm-stock-price-prediction/custom_objects')
%run "../helpers/data_manipulator.py"
%run "../helpers/yfinance_data_fetcher.py"

from custom_objects import register_custom_objects, precision_with_threshold, recall_with_threshold, focal_loss, true_positives, all_positives, recall_mul_prediction, f1_score_metric, cubic_loss

# Register globally
register_custom_objects()

In [2]:
NUM_EPOCH = 200
STARTING_LR = 0.0001
SEQUENCE_SIZE = 16
REDUCE_LR_PATIENCE = 6
STOP_L_PATIENCE = 12
BATCH_SIZE = 32
LOSS_FUNCTION = "mse"
RELOAD_FROM_NUMPY_FILE = False

target = "High-Open-Ratio"
is_binary_prediction = False

SPEC_FILE_TAG = "last_trained_model"

In [3]:
# Import necessary libraries and modules
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense, Conv1D, BatchNormalization, PReLU, LayerNormalization, Reshape, Lambda
from tensorflow.keras.regularizers import l2
from keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from keras.models import load_model
from sklearn.preprocessing import MinMaxScaler
from keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import pickle
import math
import keras.backend as K
import os
import logging
import absl.logging
import joblib
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')
# Suppress absl.logging (used internally by TensorFlow)
logging.getLogger('absl').setLevel(logging.ERROR)


### 5. TRAINING LSTM MODEL

In [4]:
stocks = limited_stocks

In [5]:
# Let's create ML-ready dataset
data_frame = construct_values_for_model(sequence_size = SEQUENCE_SIZE,
                                        ticker_symbols=stocks,
                                        use_for_last_day_prediction=False,
                                        verbose=False,
                                        refresh=False,
                                        data_interval="5y",
                                        find_sectors=True)

TypeError: construct_values_for_model() got an unexpected keyword argument 'normalize_y'

In [ ]:

train_data, validation_data, test_data = split_data(data_frame,
                                                    train_cut_date = "2024-03-31",
                                                    validate_cut_date = "2024-09-30",
                                                    test_end_date = "2024-12-31")

In [ ]:
print(train_data["y-value"])
print(train_data["y-value-original"])

In [ ]:
# BElow verifying the arr conversions work properly. check the values...
print(train_data.iloc[0]["LstmData"].iloc[0])
print(np.array(train_data["LstmData"].to_list())[0][0])


print(f"train data shape:{train_data.shape} and train data columns: {train_data.columns}")
print(f"validation_data shape:{validation_data.shape} and validation_data columns: {validation_data.columns}")
print(f"test_data shape:{test_data.shape} and test_data columns: {test_data.columns}")

#### Building LSTM Model

In [ ]:
### MODEL 1
from tensorflow.keras.layers import Concatenate, ZeroPadding1D, Cropping1D, Embedding, RepeatVector

input_data = train_data.iloc[0]["LstmData"]
print(input_data.shape)

input_layer = Input(shape=(input_data.shape[0], input_data.shape[1]), name="values")
stock_id_input = Input(shape=(1,), name="stock_id")  # Stock ID input
sector_input = Input(shape=(1,), name="sector_id")  # Sector ID input


# Stock Embedding Layer
num_of_unique_stocks = train_data["Ticker"].nunique()  # Adjust based on the number of unique stocks
num_of_unique_sectors = train_data["Sector"].nunique()  # Adjust based on the number of unique stocks

# Roll to the nearest higher integer
stock_embedding_dim = 512 #math.ceil(math.sqrt(num_of_unique_stocks)) * 2
sector_embedding_dim = 128 #math.ceil(math.sqrt(num_of_unique_sectors)) * 2

stock_embedding = Embedding(input_dim=num_of_unique_stocks, output_dim=stock_embedding_dim, name="stock_embedding")(stock_id_input)
sector_embedding = Embedding(input_dim=num_of_unique_sectors, output_dim=sector_embedding_dim, name="sector_embedding")(sector_input)

# Reshape to remove the sequence length dimension
stock_embedding = Reshape((stock_embedding_dim,))(stock_embedding)
sector_embedding = Reshape((sector_embedding_dim,))(sector_embedding)

cnn = Conv1D(128,3, kernel_regularizer=l2(0.00001))(input_layer)
cnn = PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.25))(cnn)
cnn = Conv1D(128,3, kernel_regularizer=l2(0.00001))(cnn)
cnn = PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.25))(cnn)
cnn = Conv1D(256,3, kernel_regularizer=l2(0.00001))(cnn)
cnn = PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.25))(cnn)
cnn = Conv1D(256,3, kernel_regularizer=l2(0.00001))(cnn)
cnn = PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.25))(cnn)
cnn = Conv1D(512,3, kernel_regularizer=l2(0.00001))(cnn)
cnn = PReLU(shared_axes=[1], alpha_initializer=tf.initializers.constant(0.25))(cnn)

# Pass embeddings through a dense layer
stock_embedding = Dense(128, activation='relu', name="stock_dense")(stock_embedding)
sector_embedding = Dense(128, activation='relu', name="sector_dense")(sector_embedding)

stock_embedding = Dropout(0.2)(stock_embedding)
sector_embedding = Dropout(0.2)(sector_embedding)

expanded_stock_embedding = RepeatVector(cnn.shape[1])(stock_embedding)  # Match timesteps with input data
cnn = Concatenate(axis=-1)([cnn, expanded_stock_embedding])

expanded_sector_embedding = RepeatVector(cnn.shape[1])(sector_embedding)  # Match timesteps with input data
cnn = Concatenate(axis=-1)([cnn, expanded_sector_embedding])

b = 1.2
a = 0.90

lstm = LSTM(units = 800, return_sequences = True)(cnn)
lstm = Dropout(rate = 0.3)(lstm)
lstm = LSTM(units = 400, return_sequences = True)(cnn)
lstm = Dropout(rate = 0.3)(lstm)
lstm = LSTM(units = 400, return_sequences = True)(cnn)
lstm = Dropout(rate = 0.3)(lstm)
lstm = LSTM(units = 100)(lstm)
lstm = Dropout(rate = 0.3)(lstm)
lstm = Dense(1, activation='tanh')(lstm)

In [ ]:
regressor = Model([input_layer, stock_id_input, sector_input],lstm)
regressor.summary()

from tensorflow.keras.utils import plot_model
plot_model(regressor, to_file='regressor_tree.png', show_shapes=True, show_layer_names=True)

### Compile the model

In [ ]:
optimizer = Adam(learning_rate=STARTING_LR)
loss = None
loss_name = None

if LOSS_FUNCTION == "mse":
    loss = "mse"
    loss_name = "mse"
elif LOSS_FUNCTION == "focal":
    loss = focal_loss(alpha=0.25, gamma=2.0)
    loss_name = "focal_loss"
elif LOSS_FUNCTION == "mae":
    loss = "mae"
    loss_name = "mae"
elif LOSS_FUNCTION == "cubic":
    loss = cubic_loss(power=3)
    loss_name="cubic"
elif LOSS_FUNCTION == "square":
    loss = cubic_loss(power=0.5)
    loss_name="square"
else:
    loss = "binary_crossentropy"
    loss_name = "binary_crossentropy"

regressor.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=[
        precision_with_threshold(0.7),  # Custom precision
        recall_with_threshold(0.7),    # Custom recall
        'AUC',                         # Built-in AUC
        'Precision',                   # Built-in Precision
        'Recall',                      # Built-in Recall
        f1_score_metric,                # Custom F1 score
        true_positives,
        all_positives,
        recall_mul_prediction
    ]
)

### Adjust parameters for the training

In [ ]:
file_name_variable = f"special_{SPEC_FILE_TAG}_epoch_{NUM_EPOCH}_lr{STARTING_LR}_batch{BATCH_SIZE}_seqsize{SEQUENCE_SIZE}_loss{loss_name}"

# Create a checkpoint to monitor the validation loss and save the model with the best performance.
model_location = "..//models//"
model_name = f"model_{file_name_variable}.keras"
best_model_checkpoint_callback = ModelCheckpoint(
    model_location + model_name, 
    monitor="val_loss", 
    save_best_only=True, 
    mode="min", 
    verbose=0)

# Create the ReduceLROnPlateau callback
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',     # Metric to monitor
    factor=0.333,             # Factor to reduce the learning rate (new_lr = lr * factor)
    patience=REDUCE_LR_PATIENCE,             # Number of epochs with no improvement before reducing
    min_lr=1e-11,# Minimum learning rate
    verbose=1
)

# EarlyStopping to stop training
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=STOP_L_PATIENCE,
    restore_best_weights=True,
    min_delta=0.00001
)

if NORMALIZE_Y is False:
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=train_data["y-value"].unique(),
        y=train_data["y-value"].to_list()
    )
    class_weights = dict(enumerate(class_weights))

In [ ]:
import logging

# Configure logging
logging.basicConfig(
    filename=f"training_output_{file_name_variable}.log", 
    level=logging.INFO, 
    format='%(asctime)s - %(message)s'
)
logger = logging.getLogger()

# Define a custom callback to log training progress
class LogCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logger.info(f"Epoch {epoch + 1}: {logs}")


### START THE TRAINING

In [ ]:
### Let's convert all of our stuff to np arrays to avoid any indexing issues:

X_train = np.array(train_data["LstmData"].to_list())
X_train_ticker = np.array(train_data["Ticker"].to_list())
X_train_sector = np.array(train_data["Sector"].to_list())
y_train = np.array(train_data["y-value"].to_list())

X_validate = np.array(validation_data["LstmData"].to_list())
X_validate_ticker = np.array(validation_data["Ticker"].to_list())
X_validate_sector = np.array(validation_data["Sector"].to_list())
y_validate = np.array(validation_data["y-value"].to_list())


print(X_train.shape)
print(y_train.shape)
print(f"total number of positives in train data: {np.sum(y_train)}")

print(X_validate.shape)
print(y_validate.shape)
print(f"total number of positives in validate data: {np.sum(y_validate)}")

In [ ]:
import datetime

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)


In [ ]:
import keras

keras.config.disable_traceback_filtering()

# Training the model
history = regressor.fit(
    [X_train, X_train_ticker, X_train_sector]
    ,y_train
    ,validation_data=([X_validate, X_validate_ticker, X_validate_sector], y_validate)
    ,epochs=NUM_EPOCH
    ,batch_size = BATCH_SIZE
    ,callbacks = [best_model_checkpoint_callback, reduce_lr, early_stopping, LogCallback(), tensorboard_callback]
    # ,class_weight=class_weights
)

regressor.load_weights(model_location + model_name)

#### Performance Evaluation

In [ ]:
# Visualizing model performance during training
plt.figure(figsize=(18, 6))

plt.plot(history.history["loss"][1:], label="Training Loss")
plt.plot(history.history["val_loss"][1:], label="Validation Loss")

plt.title("LSTM Model Performance")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

### LET'S TRY THIS ON THE TEST DATA NOW

In [ ]:
# y_test_predict = regressor.predict([X_test, X_test_ticker, X_test_sector])

y_validate_predict = regressor.predict([X_validate, X_validate_ticker, X_validate_sector])

### SANITATION:

In [ ]:
validation_data['y-predict'] = y_validate_predict
validation_data['y-predict-original'] = inverse_normalize_data_for_y(validation_data[['y-predict']], ["y-predict"])

### LET'S SHOW THESE FALSE POSITIVES AND CHECK FOR TRENDS

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Define chart colors
train_actual_color = "cornflowerblue"
validate_actual_color = "orange"
test_actual_color = "green"
train_predicted_color = "lightblue"
validate_predicted_color = "peru"
test_predicted_color = "limegreen"


In [ ]:
ticker_to_show = 0
ticker_data = validation_data[validation_data['Ticker'] == ticker_to_show]

plt.figure(figsize=(18,6))
plt.plot(ticker_data['Date'], ticker_data['y-predict'], label=f"{ticker_to_show} Prediction Values", color=train_predicted_color, marker='o', linestyle='')
plt.plot(ticker_data['Date'], ticker_data['y-original'], label=f"{ticker_to_show} Real Values", color=train_predicted_color, marker='x', linestyle='')
plt.title("Test results")
plt.xlabel("Time")
plt.ylabel("Prediction")
plt.xticks(rotation=45)
plt.legend()
plt.grid(color="lightgray")

In [ ]:
# Plot actual and predicted price
plt.figure(figsize=(18,6))
plt.plot(ticker_data['Date'], ticker_data['y-predict-original'] - ticker_data['y-value-original'], label=f"{ticker_to_show} Over-Prediction Values", color=train_predicted_color, marker='o', linestyle='')

plt.title("Test results")
plt.xlabel("Time")
plt.ylabel("Over-Prediction")
plt.xticks(rotation=45)
plt.legend()
plt.grid(color="lightgray")

In [ ]:
def calculate_for(val):

    validation_data['y-predict-binary'] = validation_data['y-predict-original'].apply(lambda x: 1 if x > val else 0)
    validation_data['y-value-binary'] = validation_data['y-value-original'].apply(lambda x: 1 if x > val else 0)


    validation_data['true-positive'] = (validation_data["y-predict-binary"] == 1.0) & (validation_data["y-value-binary"] == 1.0)
    validation_data['false-positive'] = (validation_data["y-predict-binary"] == 1.0) & (validation_data["y-value-binary"] == 0.0)
    validation_data['false-negative'] = (validation_data["y-predict-binary"] == 0.0) & (validation_data["y-value-binary"] == 1.0)

    ticker_data = validation_data[validation_data['Ticker'] == ticker_to_show]
    
    # Plot actual and predicted price
    plt.figure(figsize=(18,6))
    plt.plot(ticker_data['y-predict-binary'], label=f"{ticker_to_show} Prediction Values", color=train_predicted_color, marker='o', linestyle='')
    plt.plot(ticker_data['y-value-binary'], label=f"{ticker_to_show} Real Values", color=train_predicted_color, marker='x', linestyle='')
    plt.title("Test results")
    plt.xlabel("Time")
    plt.ylabel("Binary Prediction - True/False positives")
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(color="lightgray")

    tps = validation_data['true-positive'].sum()
    fps = validation_data['false-positive'].sum()
    fns = validation_data['false-negative'].sum()
    real_trues = validation_data['y-value-binary'].sum()
    all_values = len(validation_data)
    
    print(f"Number of true positives for the given ticker: {tps}")
    print(f"Number of false positives for the given ticker: {fps}")
    print(f"Precision score: {tps/ (tps+fps)}")
    print(f"Recall score: {tps/ (tps+fns)}")
    print(f"Random guess score: {real_trues/ all_values}")
    
    tp_per_ticker = validation_data.groupby('Orig_Ticker')['true-positive'].sum()
    fp_per_ticker = validation_data.groupby('Orig_Ticker')['false-positive'].sum()
    fn_per_ticker = validation_data.groupby('Orig_Ticker')['false-negative'].sum()
    
    print(f"true-positive per ticker: {tp_per_ticker}")
    print(f"false-positive per ticker: {fp_per_ticker}")
    print(f"false-negative per ticker: {fn_per_ticker}")
    print(f"precision per ticker: {tp_per_ticker / (tp_per_ticker + fp_per_ticker)}")
    print(f"recall per ticker: {tp_per_ticker / (tp_per_ticker + fn_per_ticker)}")
    print(f"rand guess per ticker: {(tp_per_ticker + fn_per_ticker) / len(ticker_data)}")

    print(f"Some true positives")
    print(validation_data[validation_data['true-positive'] == 1.0][["Date", "Orig_Ticker", "y-predict", "y-value-original"]].sample(n=5, random_state=42))

    print(f"Some false positives")
    print(validation_data[validation_data['false-positive'] == 1.0][["Date", "Orig_Ticker", "y-predict", "y-value-original"]].sample(n=5, random_state=42))

In [ ]:
calculate_for(1.02)

In [ ]:

calculate_for(1.021)

In [ ]:

calculate_for(1.022)

In [ ]:
calculate_for(1.023)

In [ ]:
calculate_for(1.024)

In [ ]:

calculate_for(1.025)

In [ ]:

calculate_for(1.015)

In [ ]:

calculate_for(1.026)

In [ ]:

calculate_for(1.027)